# NVIDIA Financial News: Dataset Investigation

## Project overview

This notebook investigates the **Tech-Stocks-News** dataset and extracts news headlines that mention NVIDIA. The goal at this stage is data investigation and preparation—not stock-price prediction.

We will:
- inspect the source dataset and its schema;
- stream through the records rather than downloading the entire dataset into memory;
- identify NVIDIA-related headlines using company-name and ticker-symbol keywords;
- examine the date coverage and yearly distribution;
- inspect repeated records and headline duplication;
- export the candidate records and a deduplicated version.

**Important:** A keyword match is only a candidate. A headline can mention NVIDIA without being mainly about NVIDIA. The extracted data therefore needs relevance checks before it is used for sentiment analysis or financial modelling.

## Dataset source and contents

We use [`David-Chew-HL/Tech-Stocks-News`](https://huggingface.co/datasets/David-Chew-HL/Tech-Stocks-News) hosted on the Hugging Face Hub.

According to the dataset card, the collection contains news items/headlines related to several technology companies, including **AAPL, AMZN, TSLA, GOOG, MSFT, META, BABA, and NVDA**. The dataset card reports a date range from **2015-01-01 to 2024-01-30**, **276,558 records**, and a total file size of approximately **912 MB**. It also says the data was pulled from the **Alpaca Markets API**. The schema preview exposes two fields:
- `headline`: text of the news item;
- `date`: publication date as a string.

This notebook checks the live dataset when it runs; counts printed below may differ if the Hub dataset changes.

**Source and licensing note:** The Hugging Face page displays an `apache-2.0` license label. Because the dataset contains news text originating from third-party publishers, check the dataset card and the original publishers' terms before redistributing the text or using it commercially. A repository-level license label should not automatically be treated as permission to republish every underlying article or headline.

## Scope and terminology

In this notebook, *candidate record* means a row whose headline contains `NVIDIA` or `NVDA` as a whole word. This is a broad text filter, not a human-verified relevance label. The broader product/technology keyword scan later in the notebook reports keyword coverage only; it does **not** append those additional matches to `df_nvda` or to the exported CSV.


## 1. Install and import dependencies

The notebook uses:
- **Hugging Face `datasets`** to access the dataset hosted on the Hub;
- **pandas** to create and inspect tables, parse dates, and identify duplicates;
- **matplotlib** to visualize the number of candidate headlines per year;
- Python's **`re`** module for whole-word keyword matching, and **`Counter`** for counting keyword occurrences.

The installation cell is useful in a fresh Google Colab runtime. If these packages are already installed locally, it can be skipped.


In [ ]:
!pip install -q datasets pandas matplotlib

In [ ]:
# ============================================================
# 2. IMPORT LIBRARIES
# ============================================================

import re
import pandas as pd
import matplotlib.pyplot as plt

from collections import Counter
from datasets import load_dataset

# Make pandas output easier to inspect
pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_columns", None)

## 2. Load the dataset with streaming

The source is relatively large (the dataset card reports about 912 MB). We use `streaming=True` so records can be read progressively instead of downloading and loading the complete dataset into memory at once.

A streamed dataset is an iterator: after consuming it, create a fresh stream before scanning it again. That is why the notebook loads the dataset again in the next major step.


In [ ]:
# ============================================================
# 3. LOAD DATASET
# ============================================================

DATASET_NAME = "David-Chew-HL/Tech-Stocks-News"

dataset = load_dataset(
    DATASET_NAME,
    split="train",
    streaming=True
)

print("Dataset loaded successfully!")

## 3. Inspect the schema and sample records

Before filtering, inspect a small sample to confirm the available field names and the shape of the text/date values. This notebook expects `headline` and `date`.

This is a quick schema sanity check, not a complete data-quality assessment. The later date-validation step checks the date field across the NVIDIA candidate records.


In [ ]:
# ============================================================
# 4. INSPECT DATASET STRUCTURE
# ============================================================

# Retrieve the first 10 records
sample = list(dataset.take(10))

# Display available columns
print("Available columns:")
print(sample[0].keys())

# Display sample records
print("\nFirst 5 records:")

for i, record in enumerate(sample[:5], start=1):

    print(f"\nRecord {i}")
    print("Headline:", record["headline"])
    print("Date:", record["date"])

## 4. Filter headlines that mention NVIDIA

We scan the full stream and keep a record when its headline contains either:
- `NVIDIA` as a whole word; or
- `NVDA` as a whole word (the stock ticker).

The regular-expression token boundary `\b` helps avoid matching these strings as part of longer words. Matching is case-insensitive because the headline is converted to lowercase before searching.

We also record three Boolean flags:
- `has_nvidia`: the headline contains the word `NVIDIA`;
- `has_nvda`: the headline contains the ticker `NVDA`;
- `has_full_name`: the headline contains the phrase `NVIDIA Corporation`.

The three keyword counts can overlap. For example, a headline containing both `NVIDIA` and `NVDA` contributes to both keyword counts but is stored only once in `nvidia_records` during this pass.

**Limitation:** This filter can include headlines where NVIDIA is only mentioned in passing, and it can miss relevant stories that mention only a product or architecture without naming NVIDIA or its ticker. We investigate additional terms separately later.


In [ ]:
# Reload the dataset because the earlier sample inspection consumed a stream.
dataset = load_dataset(DATASET_NAME, split="train", streaming=True)

keyword_counts = Counter()
nvidia_records = []
total_records = 0
invalid_dates = 0

for record in dataset:
    total_records += 1

    headline = str(record.get("headline") or "")
    date = str(record.get("date") or "")
    text = headline.lower()

    # Whole-word matches avoid matching "nvidia" or "nvda" inside longer words.
    has_nvidia = bool(re.search(r"\bnvidia\b", text))
    has_nvda = bool(re.search(r"\bnvda\b", text))
    has_full_name = bool(re.search(r"\bnvidia corporation\b", text))

    # Counts overlap: one headline can match more than one expression.
    keyword_counts["NVIDIA"] += has_nvidia
    keyword_counts["NVDA"] += has_nvda
    keyword_counts["NVIDIA Corporation"] += has_full_name

    # Keep a row once if it matches either company name or ticker.
    if has_nvidia or has_nvda:
        parsed_date = pd.to_datetime(date, errors="coerce")
        if pd.isna(parsed_date):
            invalid_dates += 1

        nvidia_records.append({
            "date": date,
            "headline": headline,
            "has_nvidia": has_nvidia,
            "has_nvda": has_nvda,
            "has_full_name": has_full_name
        })

    if total_records % 25000 == 0:
        print(
            f"Processed {total_records:,} source rows | "
            f"NVIDIA candidate rows: {len(nvidia_records):,}"
        )

print("\n========== PRIMARY FILTER COMPLETED ==========")
print("Total source records scanned:", total_records)
print("\nKeyword match counts (overlapping):")
for keyword, count in keyword_counts.items():
    print(f"{keyword}: {count:,}")

print("\nNVIDIA candidate rows before deduplication:", len(nvidia_records))
print("Candidates with invalid dates:", invalid_dates)


## 5. Convert the extracted records into a DataFrame

`nvidia_records` is a list of dictionaries collected during the stream. `pd.DataFrame(...)` turns it into a table so that we can use pandas for date analysis, duplicate checks, sampling, and export.

At this point, the DataFrame contains all rows that matched the company-name/ticker filter. It has not yet been deduplicated.


In [ ]:
# ============================================================
# 6. CREATE NVIDIA DATAFRAME
# ============================================================

df_nvda = pd.DataFrame(nvidia_records)

print("Dataset shape:", df_nvda.shape)

print("\nDataset information:")

df_nvda.info()

print("\nFirst 10 records:")

display(df_nvda.head(10))

## 6. Analyze publication dates and yearly coverage

The original `date` field is a string. We convert it to pandas datetime using `errors="coerce"`:
- valid date strings become datetime values;
- invalid or missing values become `NaT` (pandas' missing-datetime marker).

We then extract the calendar year and count candidate rows per year. This helps reveal gaps and uneven coverage. The counts at this stage include repeated rows; later sections quantify duplicates and produce a cleaned export.


In [ ]:
# ============================================================
# 7. ANALYZE HISTORICAL COVERAGE
# ============================================================

# Convert date column into datetime format
df_nvda["parsed_date"] = pd.to_datetime(
    df_nvda["date"],
    errors="coerce"
)

# Extract publication year
df_nvda["year"] = df_nvda["parsed_date"].dt.year

# Count records per year
yearly_counts = (
    df_nvda
    .groupby("year")
    .size()
    .sort_index()
)

print("NVIDIA candidate records per year:")

display(
    yearly_counts.rename("article_count").to_frame()
)

## 7. Visualize yearly coverage

The bar chart makes changes in the number of NVIDIA keyword-matching records easier to see.

A larger bar does not necessarily mean NVIDIA received more media attention in the real world. It may also reflect changes in source coverage, collection practices, duplicate frequency, or the dataset's composition. Interpret the chart together with the duplicate and date-quality checks.


In [ ]:
# ============================================================
# 8. VISUALIZE NEWS DISTRIBUTION
# ============================================================

plt.figure(figsize=(13, 6))

yearly_counts.plot(kind="bar")

plt.title("NVIDIA-Related News Distribution by Year")

plt.xlabel("Year")

plt.ylabel("Number of Records")

plt.xticks(rotation=45)

plt.grid(axis="y", alpha=0.3)

plt.tight_layout()

plt.show()

## 8. Manually inspect a reproducible random sample

A random sample of up to 30 candidate headlines is displayed. `random_state=42` makes the sample reproducible when the underlying DataFrame has not changed.

Read these headlines to assess whether the keyword filter is capturing genuinely NVIDIA-relevant news or merely incidental mentions. This is a qualitative spot check, not a formal relevance evaluation.


In [ ]:
# ============================================================
# 9. INSPECT RANDOM RECORDS
# ============================================================

# Select up to 30 random candidate records
random_articles = df_nvda.sample(
    n=min(30, len(df_nvda)),
    random_state=42
)

# Display publication date and headline
display(
    random_articles[
        [
            "date",
            "headline",
            "has_nvidia",
            "has_nvda"
        ]
    ]
)

## 9. Investigate duplicate records and repeated headlines

There are several different notions of duplication:
1. **Exact duplicate rows:** every column has the same value.
2. **Same date + normalized headline:** the publication date and headline match after lowercasing and collapsing whitespace.
3. **Same normalized headline on any date:** the headline text matches, even if the date differs.

These checks answer different questions. Repeated headlines can be syndicated or reposted, while exact repeated rows may be duplicate records in the dataset. The code displays duplicate examples with all occurrences included, rather than showing only the later copies.


In [ ]:
# Normalize headlines: lowercase and collapse repeated whitespace.
df_nvda["normalized_headline"] = (
    df_nvda["headline"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# 1) Exact duplicate rows: all columns match.
exact_duplicate_mask = df_nvda.duplicated(keep=False)
print("Rows belonging to exact-duplicate groups (keep=False):", exact_duplicate_mask.sum())
print("Extra exact duplicate copies (default keep=first):", df_nvda.duplicated().sum())

# 2) Same publication date + normalized headline.
date_headline_duplicate_mask = df_nvda.duplicated(
    subset=["date", "normalized_headline"],
    keep=False
)
print(
    "Rows belonging to duplicate date + headline groups:",
    date_headline_duplicate_mask.sum()
)

# 3) Same normalized headline regardless of date.
headline_duplicate_mask = df_nvda.duplicated(
    subset=["normalized_headline"],
    keep=False
)
print(
    "Rows belonging to repeated-headline groups (any date):",
    headline_duplicate_mask.sum()
)

# Display a few groups, including the first occurrence and its repeats.
duplicate_examples = (
    df_nvda.loc[date_headline_duplicate_mask]
    .sort_values(["date", "normalized_headline"])
    .groupby(["date", "normalized_headline"], dropna=False, group_keys=False)
    .head(3)
    .head(15)
)

display(duplicate_examples[["date", "headline"]])


## 10. Validate the date range and year coverage

This section reports the number of valid and invalid dates, the earliest and latest valid dates, and any missing years in the expected 2015–2024 period.

The expected period is based on the dataset card's stated coverage. A year appearing in the data does not mean every month in that year is represented. For serious time-series analysis, inspect coverage by month and check for long gaps.


In [ ]:
print("Total NVIDIA candidate rows:", len(df_nvda))
print("Valid dates:", df_nvda["parsed_date"].notna().sum())
print("Invalid or missing dates:", df_nvda["parsed_date"].isna().sum())

print("\nEarliest valid publication date:")
print(df_nvda["parsed_date"].min())

print("\nLatest valid publication date:")
print(df_nvda["parsed_date"].max())

# Check whether any calendar years are absent in the expected 2015–2024 range.
expected_years = range(2015, 2025)
existing_years = set(df_nvda["year"].dropna().astype(int))
missing_years = [year for year in expected_years if year not in existing_years]

print("\nYears without NVIDIA candidate records:")
print(missing_years)

# Monthly counts help reveal gaps that a yearly summary can hide.
valid_date_rows = df_nvda.dropna(subset=["parsed_date"]).copy()
monthly_counts = (
    valid_date_rows
    .groupby(valid_date_rows["parsed_date"].dt.to_period("M"))
    .size()
)

print("\nMost recent 15 months represented:")
display(monthly_counts.tail(15).rename("candidate_rows").to_frame())


## 11. Export the NVIDIA candidate records

This exports the candidate DataFrame before the final deduplication step. It is useful as an audit copy of what the keyword filter returned.

The file contains keyword flags and date-analysis columns created in this notebook. It is not the untouched original source dataset, and it is not a manually validated NVIDIA-only news corpus.


In [ ]:
# ============================================================
# 12. EXPORT NVIDIA CANDIDATES
# ============================================================

# Save all potential NVIDIA-related records
df_nvda.to_csv(
    "nvidia_news_candidates.csv",
    index=False
)

print("CSV exported successfully!")

### Optional: download the candidate CSV (Google Colab)

This cell downloads the candidate CSV in Colab. Outside Colab, the file remains in the notebook's working directory.

In [ ]:
# Optional: download the candidate CSV directly when running in Google Colab.
# When running locally, the file is saved in the current working directory.
try:
    from google.colab import files
    files.download("nvidia_news_candidates.csv")
except ImportError:
    print("Not running in Google Colab. Find nvidia_news_candidates.csv in the current working directory.")


## 12. Final investigation report

The report gathers the main counts and date checks into a compact table and saves them as a separate CSV. It is generated from the current notebook run, so rerun it whenever the source dataset or filtering logic changes.


In [ ]:
# Build a compact report from the current run.
report = {
    "Source dataset": DATASET_NAME,
    "Total source records scanned": total_records,
    "NVIDIA keyword matches": keyword_counts["NVIDIA"],
    "NVDA keyword matches": keyword_counts["NVDA"],
    "NVIDIA Corporation keyword matches": keyword_counts["NVIDIA Corporation"],
    "NVIDIA candidate rows (before deduplication)": len(df_nvda),
    "Valid dates": int(df_nvda["parsed_date"].notna().sum()),
    "Invalid or missing dates": int(df_nvda["parsed_date"].isna().sum()),
    "Rows in exact-duplicate groups (keep=False)": int(df_nvda.duplicated(keep=False).sum()),
    "Extra exact duplicate copies": int(df_nvda.duplicated().sum()),
    "Rows in duplicate date + headline groups": int(
        df_nvda.duplicated(
            subset=["date", "normalized_headline"],
            keep=False
        ).sum()
    ),
    "Earliest valid date": str(df_nvda["parsed_date"].min()),
    "Latest valid date": str(df_nvda["parsed_date"].max()),
    "Missing years (2015–2024)": missing_years
}

report_df = pd.DataFrame(
    list(report.items()),
    columns=["Metric", "Result"]
)

display(report_df)

report_df.to_csv(
    "nvidia_dataset_investigation_report.csv",
    index=False
)


## 13. Broader NVIDIA keyword coverage (counts only)

The next scan checks additional NVIDIA-related names and technologies, grouped into:
- company identifiers;
- people and NVIDIA technologies;
- GPU architectures and hardware models.

This is an exploratory coverage check. The counts show how often each expression appears in headlines across the source dataset. They are **not** counts of unique articles, and they overlap: one headline may contain several terms.

**Important:** This section only counts matches. It does not add these records to `df_nvda`, does not create an expanded candidate DataFrame, and does not change the CSV exported above.


In [ ]:
# Define keyword groups for a broader scan of the source headlines.
# These are investigated separately from the primary NVIDIA/NVDA filter.

# Category 1: Company identifiers
company_keywords = {
    "NVIDIA": r"\bnvidia\b",
    "NVDA": r"\bnvda\b",
    "NVIDIA Corporation": r"\bnvidia corporation\b"
}

# Category 2: People and NVIDIA technologies
product_keywords = {
    "Jensen Huang": r"\bjensen huang\b",
    "GeForce": r"\bgeforce\b",
    "CUDA": r"\bcuda\b",
    "DGX": r"\bdgx\b",
    "Jetson": r"\bjetson\b",
    "TensorRT": r"\btensorrt\b",
    "Omniverse": r"\bomniverse\b",
    "NVLink": r"\bnvlink\b"
}

# Category 3: Architectures and GPU models
hardware_keywords = {
    "Blackwell": r"\bblackwell\b",
    "Hopper": r"\bhopper\b",
    "Ampere": r"\bampere\b",
    "H100": r"\bh100\b",
    "H200": r"\bh200\b",
    "B100": r"\bb100\b",
    "B200": r"\bb200\b",
    "A100": r"\ba100\b"
}

all_categories = {
    "Company": company_keywords,
    "People and technologies": product_keywords,
    "Hardware and architectures": hardware_keywords
}

keyword_counts = Counter()
category_counts = Counter()
total_records = 0

# Start a fresh stream because the earlier stream was already consumed.
dataset = load_dataset(DATASET_NAME, split="train", streaming=True)

for record in dataset:
    total_records += 1
    headline = str(record.get("headline") or "")

    matched_categories = set()

    for category, keywords in all_categories.items():
        for keyword, pattern in keywords.items():
            if re.search(pattern, headline, re.IGNORECASE):
                keyword_counts[keyword] += 1
                matched_categories.add(category)

    # Count each source row at most once in each category.
    for category in matched_categories:
        category_counts[category] += 1

    if total_records % 25000 == 0:
        print(f"Scanned: {total_records:,} records")

print("\n========== BROADER KEYWORD INVESTIGATION ==========")
print("Total source records scanned:", total_records)

print("\nKeyword match counts (a row may match multiple keywords):")
for keyword, count in keyword_counts.most_common():
    print(f"{keyword}: {count:,}")

print("\nCategory coverage (each row counted once per category):")
for category, count in category_counts.items():
    print(f"{category}: {count:,}")


## 14. Create a deduplicated candidate CSV

We create a separate DataFrame so that `df_nvda` remains unchanged. For this export, duplicates are defined as rows with the same `date` and the same normalized headline (lowercase, with repeated whitespace collapsed).

This is a practical deduplication key for this dataset, but it is not perfect:
- identical headlines on different dates are retained;
- slightly different headline wording about the same event is not detected;
- syndicated stories with different headlines may still describe the same event.

The output is a new CSV file; the source dataset and `df_nvda` are not overwritten.


In [ ]:
# Work on a copy so the extracted candidate DataFrame stays unchanged.
df_final = df_nvda.copy()

# Normalize headline text for a consistent comparison.
df_final["normalized_headline"] = (
    df_final["headline"]
    .fillna("")
    .astype(str)
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Count repeated date + headline rows (this excludes the first occurrence
# in each duplicate group, which is the number of rows that will be removed).
duplicates_to_remove = df_final.duplicated(
    subset=["date", "normalized_headline"]
).sum()

# Keep the first occurrence of each date + normalized-headline pair.
df_final = df_final.drop_duplicates(
    subset=["date", "normalized_headline"],
    keep="first"
).copy()

# Remove the temporary comparison column from the exported file.
df_final.drop(columns=["normalized_headline"], inplace=True)
df_final.reset_index(drop=True, inplace=True)

# Export to a new CSV. This does not overwrite the original source or df_nvda.
output_file = "nvidia_news_candidates_unique.csv"
df_final.to_csv(output_file, index=False)

print("========== EXPORT RESULTS ==========")
print("Candidate rows before deduplication:", len(df_nvda))
print("Rows removed by date + headline rule:", duplicates_to_remove)
print("Rows in deduplicated export:", len(df_final))
print("Output file:", output_file)


### Optional: download the deduplicated CSV (Google Colab)

This cell downloads the cleaned CSV in Colab. Outside Colab, the file remains in the notebook's working directory.

In [ ]:
# Optional: download the deduplicated CSV directly when running in Google Colab.
try:
    from google.colab import files
    files.download("nvidia_news_candidates_unique.csv")
except ImportError:
    print("Not running in Google Colab. Find nvidia_news_candidates_unique.csv in the current working directory.")


## 15. Yearly coverage after exact-row deduplication

This final check removes rows that are identical across all columns, then reports the remaining number of rows per year.

Notice that this is not necessarily the same as the date-plus-normalized-headline deduplication used for the final CSV above. The two methods use different duplicate definitions, so their resulting row counts can differ.


In [ ]:
# Remove rows that are identical across every column.
df_nvda_clean = df_nvda.drop_duplicates().copy()

print("Total candidate rows:", len(df_nvda))
print("Unique rows after exact-row deduplication:", len(df_nvda_clean))
print("Extra exact duplicate copies:", df_nvda.duplicated().sum())

print("\nUnique candidate rows per year:")
display(
    df_nvda_clean["year"]
    .value_counts()
    .sort_index()
    .rename("unique_rows")
    .to_frame()
)


# 16. Checklist before using financial news data

Before moving from data preparation to sentiment analysis or stock-market research, check the following.

## Source, provenance, and licensing
- [ ] Record the dataset name, Hugging Face URL, access date, and (ideally) a fixed dataset revision/commit so the analysis can be reproduced.
- [ ] Read the dataset card and understand the stated source (the card says Alpaca Markets API).
- [ ] Check the dataset's license information and the terms that apply to the underlying news publishers before redistributing text or using it commercially.
- [ ] Keep a note of the original dataset fields and every transformation applied.

## Coverage and relevance
- [ ] Confirm the actual minimum and maximum dates in the downloaded/streamed data.
- [ ] Inspect monthly coverage, not only yearly totals; identify missing periods and sudden changes in volume.
- [ ] Check whether the dataset covers the full period needed for the research question.
- [ ] Manually review a random sample of matched headlines.
- [ ] Estimate how many matches are genuinely about NVIDIA versus headlines that mention NVIDIA only in passing.
- [ ] Review false positives from ambiguous terms such as `Ampere`, `Hopper`, or model names. Do not use these terms alone as proof that an article is about NVIDIA.

## Duplicates and data quality
- [ ] Report exact duplicate rows separately from duplicate date/headline pairs and repeated headlines across different dates.
- [ ] Choose and document a deduplication key. Do not silently remove records.
- [ ] Inspect examples before deleting duplicates; syndicated or updated stories may be meaningful repeats.
- [ ] Check missing headlines, empty strings, malformed dates, and inconsistent date formats.
- [ ] Confirm that the cleaned export has the expected number of rows and that the original extracted DataFrame remains unchanged.

## Financial-news and NLP considerations
- [ ] Remember that this dataset's visible schema contains `headline` and `date`; it does not provide full article bodies, source/publisher names, sentiment labels, or NVIDIA stock prices in the fields inspected here.
- [ ] Do not treat a headline keyword match as a sentiment label or as proof of causal impact on the stock.
- [ ] Financial sentiment is context-dependent: positive wording about a competitor, for example, may not be positive for NVIDIA.
- [ ] If adding stock prices, align news publication dates/times with the correct trading calendar and market timezone. Consider pre-market, regular-session, after-hours, weekends, and holidays.
- [ ] Avoid look-ahead bias: only use information that would have been available at the prediction time.
- [ ] Use chronological train/validation/test splits for time-series experiments rather than randomly shuffling records.
- [ ] Compare results with simple baselines and report suitable metrics; do not rely on accuracy alone if classes are imbalanced.
- [ ] Treat any observed news/price relationship as an association unless the research design supports a stronger causal claim.

## Interpretation of this notebook's output

The primary extraction is a **keyword-based NVIDIA candidate dataset**, not a fully verified collection of NVIDIA-focused articles. The broader keyword scan is a separate frequency investigation and does not append extra rows to the exported candidates. The deduplicated CSV uses the date + normalized headline rule. These choices should be stated clearly in any README, report, or presentation.
